In [27]:
# CELL 1 — Installation for Mac
# ffmpeg is installed via brew in terminal (see above), not here
!pip install transformers torch torchaudio soundfile librosa speechbrain datasets accelerate jiwer -q

# Verify ffmpeg is accessible
import subprocess
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ ffmpeg is working!")
else:
    print("❌ ffmpeg not found — run: brew install ffmpeg in your terminal")

✅ ffmpeg is working!


In [19]:
from transformers import pipeline
import numpy as np
import librosa

# ─── Load the Darija ASR model (FIXED for newer transformers) ────────────────
# The model is already fine-tuned for Darija, so we don't pass language/task

asr_pipeline = pipeline(
    task="automatic-speech-recognition",
    model="KandirResearch/Whisper-Small-Darija",
    chunk_length_s=30,
    # Removed: generate_kwargs={"language": "arabic", "task": "transcribe"}
    # ↑ This was causing the error. Model already knows Darija.
)

print("✅ Model loaded successfully (compatibility fix applied)!")

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 8315.97it/s]
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✅ Model loaded successfully (compatibility fix applied)!


In [10]:
# ─── Helper: load any audio file and normalize it to 16kHz mono ──────────────
# Most ASR models require 16kHz. This function handles any input sample rate.

def load_audio(file_path: str, target_sr: int = 16000) -> dict:
    """
    Load an audio file and resample to 16kHz mono.
    Accepts: .wav, .mp3, .ogg, .m4a (needs ffmpeg for mp3/m4a)
    Returns: dict with 'array' and 'sampling_rate' keys
    """
    audio_array, sr = librosa.load(file_path, sr=target_sr, mono=True)
    print(f"  Audio loaded: {len(audio_array)/target_sr:.2f}s at {target_sr}Hz")
    return {"array": audio_array.astype(np.float32), "sampling_rate": target_sr}


# ─── Transcribe ──────────────────────────────────────────────────────────────

def transcribe(audio_path: str) -> str:
    """
    Takes a path to an audio file.
    Returns the raw transcription text in Darija/Arabic.
    """
    audio = load_audio(audio_path)
    result = asr_pipeline(audio)
    transcription = result["text"].strip()
    print(f"  📝 Transcription: {transcription}")
    return transcription


# ─── Test it ─────────────────────────────────────────────────────────────────
# Replace with your actual audio file path
# audio_path = "/content/patient_sample.wav"
# transcription = transcribe(audio_path)

In [20]:
# ─── Real-world Darija extraction for AirDwa ─────────────────────────────────
# Handles how people ACTUALLY talk in rural Morocco:
#   - Symptoms ("skhana", "koh-ha") not medication names
#   - Village names ("Tizgui", "Imlil") not station numbers
#   - Mixed Latin/Arabic script
import re
from difflib import get_close_matches

# ═════════════════════════════════════════════════════════════════════════════
# 1. SYMPTOM → MEDICATION MAPPING
# ═════════════════════════════════════════════════════════════════════════════
# Patients describe symptoms, not drugs. We map symptoms to required meds.

SYMPTOM_TO_MEDICATION = {
    # Fever
    "skhana": {"med": "paracetamol", "urgency": "routine"},
    "سخانة": {"med": "paracetamol", "urgency": "routine"},
    "fièvre": {"med": "paracetamol", "urgency": "routine"},
    # Cough / respiratory
    "koh-ha": {"med": "cough_syrup", "urgency": "routine"},
    "كحة": {"med": "cough_syrup", "urgency": "routine"},
    "rabu": {"med": "ventolin", "urgency": "high"},
    "ربو": {"med": "ventolin", "urgency": "high"},
    "asthme": {"med": "ventolin", "urgency": "high"},
    # Pain
    "wja3 ras": {"med": "doliprane", "urgency": "routine"},
    "وجع راس": {"med": "doliprane", "urgency": "routine"},
    "wja3 karch": {"med": "spasfon", "urgency": "routine"},
    "وجع كرش": {"med": "spasfon", "urgency": "routine"},
    # Diarrhea / dehydration
    "is-hal": {"med": "smecta", "urgency": "routine"},
    "إسهال": {"med": "smecta", "urgency": "routine"},
    # Chronic
    "soukar": {"med": "insulin", "urgency": "routine"},
    "سكر": {"med": "insulin", "urgency": "routine"},
    "dght": {"med": "amlodipine", "urgency": "routine"},
    "ضغط": {"med": "amlodipine", "urgency": "routine"},
    # CRITICAL emergencies
    "lda3a": {"med": "antivenom", "urgency": "critical"},
    "لدغة": {"med": "antivenom", "urgency": "critical"},
    "morsure": {"med": "antivenom", "urgency": "critical"},
    "hassasiya": {"med": "epipen", "urgency": "critical"},
    "حساسية": {"med": "epipen", "urgency": "critical"},
    "allergie": {"med": "epipen", "urgency": "critical"},
    "jarh": {"med": "first_aid_kit", "urgency": "high"},
    "جرح": {"med": "first_aid_kit", "urgency": "high"},
}

# Direct medication names (in case user names the medicine directly)
DIRECT_MEDICATIONS = [
    "doliprane", "paracetamol", "ibuprofen", "amoxicilline", "amoxicillin",
    "ventolin", "insulin", "insuline", "augmentin", "smecta", "spasfon",
    "efferalgan", "dafalgan", "antivenom", "epipen",
]


def extract_medication(text: str) -> dict:
    """
    Extracts what medication to deliver, based on either:
      1. A symptom mentioned (e.g. "skhana" → paracetamol)
      2. A direct medication name (e.g. "doliprane")
    
    Returns dict with:
      - medication: standardized name
      - urgency: routine | high | critical
      - source: 'symptom' or 'direct' or None
    """
    text_lower = text.lower()
    
    # Check symptoms first (most common in real speech)
    for symptom, info in SYMPTOM_TO_MEDICATION.items():
        if symptom.lower() in text_lower:
            return {
                "medication": info["med"],
                "urgency": info["urgency"],
                "source": "symptom",
                "matched_term": symptom,
            }
    
    # Then check direct medication names (exact match)
    for med in DIRECT_MEDICATIONS:
        if med.lower() in text_lower:
            return {
                "medication": med,
                "urgency": "routine",
                "source": "direct",
                "matched_term": med,
            }
    
    # Fuzzy fallback for ASR errors on medication names
    words = re.findall(r"\b\w{4,}\b", text_lower)
    for word in words:
        matches = get_close_matches(word, DIRECT_MEDICATIONS, n=1, cutoff=0.75)
        if matches:
            return {
                "medication": matches[0],
                "urgency": "routine",
                "source": "fuzzy",
                "matched_term": word,
            }
    
    return {"medication": None, "urgency": None, "source": None, "matched_term": None}


# ═════════════════════════════════════════════════════════════════════════════
# 2. LOCATION EXTRACTION (Village name OR station code)
# ═════════════════════════════════════════════════════════════════════════════
# In rural Morocco, villages have names like Tizgui, Imlil, Asni.
# Sometimes the system assigns numeric codes too. Handle both.

# Known douars in the High Atlas region — extend this with your real data
KNOWN_DOUARS = [
    "tizgui", "imlil", "asni", "ouirgane", "tahanaout", "amizmiz",
    "tnine ourika", "setti fatma", "tinzert", "ait barka", "armed",
    "tachdirt", "tafza", "agouns", "ighbouli",
]

LOCATION_KEYWORDS = ["douar", "dwar", "duar", "دوار", "f", "في", "من", "fi"]

def extract_location(text: str) -> dict:
    """
    Extracts delivery location. Returns:
      - location: village name (preferred) OR station code
      - type: 'village' | 'code' | None
      - confidence: high | medium | low
    """
    text_lower = text.lower()
    
    # Strategy 1: Match against known douar names (highest confidence)
    for douar in KNOWN_DOUARS:
        if douar in text_lower:
            return {
                "location": douar.title(),
                "type": "village",
                "confidence": "high",
            }
    
    # Strategy 2: Word after location keyword (e.g. "douar Tizgui" → "Tizgui")
    keyword_pattern = (
        r"(?:" + "|".join(re.escape(k) for k in LOCATION_KEYWORDS) + r")"
        r"\s+([A-Za-z\u0600-\u06FF]+)"
    )
    match = re.search(keyword_pattern, text_lower)
    if match:
        candidate = match.group(1).strip()
        # Fuzzy match against known douars
        fuzzy = get_close_matches(candidate, KNOWN_DOUARS, n=1, cutoff=0.7)
        if fuzzy:
            return {"location": fuzzy[0].title(), "type": "village", "confidence": "medium"}
        # Unknown village but follows pattern — return as-is
        return {"location": candidate.title(), "type": "village", "confidence": "low"}
    
    # Strategy 3: Numeric station code fallback
    arabic_to_western = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
    text_norm = text_lower.translate(arabic_to_western)
    code_match = re.search(r"\b([A-Za-z]?-?\d{1,4})\b", text_norm)
    if code_match:
        return {"location": code_match.group(1).upper(), "type": "code", "confidence": "medium"}
    
    return {"location": None, "type": None, "confidence": None}


# ═════════════════════════════════════════════════════════════════════════════
# 3. UNIFIED EXTRACTOR
# ═════════════════════════════════════════════════════════════════════════════

def extract_order_info(transcription: str) -> dict:
    """
    Main extraction function. Takes raw Darija transcription,
    returns structured order ready for the AirDwa dispatcher.
    """
    med_info = extract_medication(transcription)
    loc_info = extract_location(transcription)
    
    # Determine overall completeness
    is_complete = med_info["medication"] is not None and loc_info["location"] is not None
    
    return {
        "transcription": transcription,
        "medication": med_info["medication"],
        "urgency": med_info["urgency"] or "routine",
        "location": loc_info["location"],
        "location_type": loc_info["type"],
        "status": "ready_to_dispatch" if is_complete else "incomplete",
        "missing": [
            field for field, value in [
                ("medication", med_info["medication"]),
                ("location", loc_info["location"]),
            ] if value is None
        ],
        "metadata": {
            "medication_source": med_info["source"],
            "medication_matched": med_info["matched_term"],
            "location_confidence": loc_info["confidence"],
        }
    }


# ═════════════════════════════════════════════════════════════════════════════
# 4. QUICK TEST — verify it works on real phrases from the report
# ═════════════════════════════════════════════════════════════════════════════
real_phrases = [
    "Khassna dwa dyal skhana f douar Tizgui",                    # from the report!
    "خاصنا دوا ديال السخانة في دوار تيزكي",                       # Arabic version
    "عندنا واحد المريض كيعاني من ربو في دوار إمليل",              # asthma in Imlil
    "نحتاج insuline f Asni",                                     # diabetic, Asni
    "lda3a dyal hanach f douar Ouirgane",                        # SNAKEBITE — critical!
    "khassna doliprane",                                          # missing location
    "f douar Tinzert",                                            # missing medication
]

print("🧪 Testing extraction on realistic Darija phrases:\n")
for phrase in real_phrases:
    result = extract_order_info(phrase)
    print(f"📞 \"{phrase}\"")
    print(f"   → Med: {result['medication']}  |  Loc: {result['location']}  |  Urgency: {result['urgency']}")
    print(f"   → Status: {result['status']}")
    if result["missing"]:
        print(f"   → ⚠️  Missing: {', '.join(result['missing'])}")
    print()

🧪 Testing extraction on realistic Darija phrases:

📞 "Khassna dwa dyal skhana f douar Tizgui"
   → Med: paracetamol  |  Loc: Tizgui  |  Urgency: routine
   → Status: ready_to_dispatch

📞 "خاصنا دوا ديال السخانة في دوار تيزكي"
   → Med: paracetamol  |  Loc: دوار  |  Urgency: routine
   → Status: ready_to_dispatch

📞 "عندنا واحد المريض كيعاني من ربو في دوار إمليل"
   → Med: ventolin  |  Loc: ربو  |  Urgency: high
   → Status: ready_to_dispatch

📞 "نحتاج insuline f Asni"
   → Med: insulin  |  Loc: Asni  |  Urgency: routine
   → Status: ready_to_dispatch

📞 "lda3a dyal hanach f douar Ouirgane"
   → Med: antivenom  |  Loc: Ouirgane  |  Urgency: critical
   → Status: ready_to_dispatch

📞 "khassna doliprane"
   → Med: doliprane  |  Loc: None  |  Urgency: routine
   → Status: incomplete
   → ⚠️  Missing: location

📞 "f douar Tinzert"
   → Med: None  |  Loc: Tinzert  |  Urgency: routine
   → Status: incomplete
   → ⚠️  Missing: medication



In [21]:
!pip install fastapi uvicorn python-multipart requests nest-asyncio -q

In [22]:
# ─── AirDwa ASR webhook service ──────────────────────────────────────────────
# This is THE deliverable for your role:
# Receives audio → transcribes → extracts → fires webhook to dispatcher

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import tempfile
import requests
import nest_asyncio
import uvicorn
from datetime import datetime
import os

nest_asyncio.apply()  # allows running uvicorn inside Jupyter

# ─── Configuration ───────────────────────────────────────────────────────────
DISPATCHER_WEBHOOK_URL = "http://localhost:8001/create_new_order"  # ← teammate's endpoint
# Set to None to skip firing webhooks during testing
SEND_WEBHOOKS = False  # change to True when dispatcher is ready

# ─── App setup ───────────────────────────────────────────────────────────────
app = FastAPI(
    title="AirDwa ASR Service",
    description="Darija voice → structured medical order",
    version="1.0.0",
)

# Allow CORS so the WhatsApp bot or kiosk frontend can call us
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ─── Response schema ─────────────────────────────────────────────────────────
class OrderResponse(BaseModel):
    order_id: str
    transcription: str
    medication: str | None
    urgency: str
    location: str | None
    location_type: str | None
    status: str
    missing: list[str]
    webhook_sent: bool
    timestamp: str


# ─── Health check ────────────────────────────────────────────────────────────
@app.get("/")
def root():
    return {"service": "AirDwa ASR", "status": "online"}


@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": asr_pipeline is not None}


# ─── MAIN ENDPOINT: audio in → order out ─────────────────────────────────────
@app.post("/process_audio", response_model=OrderResponse)
async def process_audio(audio: UploadFile = File(...)):
    """
    Main webhook endpoint.
    Receives audio from WhatsApp bot or kiosk.
    Returns structured order + fires webhook to dispatcher.
    """
    # Validate file
    if not audio.filename.lower().endswith((".wav", ".mp3", ".ogg", ".m4a", ".webm")):
        raise HTTPException(status_code=400, detail="Unsupported audio format")
    
    # Save to temp file (transformers pipeline needs a path or array)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
        contents = await audio.read()
        tmp.write(contents)
        tmp_path = tmp.name
    
    try:
        # ─── Step 1: Transcribe with Darija ASR ──────────────────────────────
        audio_data = load_audio(tmp_path)  # from your existing Cell 3
        result = asr_pipeline(audio_data, chunk_length_s=30)
        transcription = result["text"].strip()
        
        # ─── Step 2: Extract structured info ─────────────────────────────────
        order_info = extract_order_info(transcription)
        
        # ─── Step 3: Build order ID ──────────────────────────────────────────
        order_id = f"AIRDWA-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
        
        # ─── Step 4: Fire webhook to dispatcher ──────────────────────────────
        webhook_sent = False
        if SEND_WEBHOOKS and order_info["status"] == "ready_to_dispatch":
            try:
                payload = {
                    "order_id": order_id,
                    "medication": order_info["medication"],
                    "urgency": order_info["urgency"],
                    "destination": order_info["location"],
                    "source_transcription": transcription,
                }
                requests.post(DISPATCHER_WEBHOOK_URL, json=payload, timeout=5)
                webhook_sent = True
                print(f"✅ Webhook fired for {order_id}")
            except requests.RequestException as e:
                print(f"⚠️  Webhook failed: {e}")
        
        # ─── Step 5: Return structured response ──────────────────────────────
        return OrderResponse(
            order_id=order_id,
            transcription=transcription,
            medication=order_info["medication"],
            urgency=order_info["urgency"],
            location=order_info["location"],
            location_type=order_info["location_type"],
            status=order_info["status"],
            missing=order_info["missing"],
            webhook_sent=webhook_sent,
            timestamp=datetime.now().isoformat(),
        )
    
    finally:
        # Cleanup temp file
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)


# ─── Text-only test endpoint (for teammates to test without audio) ───────────
@app.post("/process_text")
async def process_text(text: str):
    """Skip ASR, test extraction logic directly with text."""
    order_info = extract_order_info(text)
    order_info["order_id"] = f"AIRDWA-TEST-{datetime.now().strftime('%H%M%S')}"
    return order_info


print("✅ FastAPI app defined")
print("   Endpoints:")
print("   • GET  /          → health check")
print("   • POST /process_audio  → main pipeline (audio file)")
print("   • POST /process_text   → test extraction (text only)")

✅ FastAPI app defined
   Endpoints:
   • GET  /          → health check
   • POST /process_audio  → main pipeline (audio file)
   • POST /process_text   → test extraction (text only)


In [24]:
# ─── Launch server in a background thread (Jupyter-safe) ─────────────────────
import threading
import time
import uvicorn

# Global reference so we can stop the server later
_server = None
_server_thread = None

def run_server():
    """Runs uvicorn in a thread so Jupyter stays responsive."""
    global _server
    config = uvicorn.Config(
        app=app,
        host="0.0.0.0",
        port=8000,
        log_level="warning",  # less noise in the notebook
    )
    _server = uvicorn.Server(config)
    _server.run()


def start_server():
    """Start the FastAPI server in a background thread."""
    global _server_thread
    
    if _server_thread is not None and _server_thread.is_alive():
        print("⚠️  Server already running on port 8000")
        return
    
    _server_thread = threading.Thread(target=run_server, daemon=True)
    _server_thread.start()
    time.sleep(2)  # give it a moment to boot up
    print("✅ AirDwa ASR server is LIVE")
    print()
    print("🌐 URLs:")
    print("   • Health check  : http://localhost:8000/")
    print("   • API docs (try it!) : http://localhost:8000/docs")
    print("   • Process audio : POST http://localhost:8000/process_audio")
    print("   • Process text  : POST http://localhost:8000/process_text")


def stop_server():
    """Stop the running server."""
    global _server
    if _server:
        _server.should_exit = True
        print("🛑 Server stopping...")
    else:
        print("ℹ️  No server is running")


# Launch it
start_server()

✅ AirDwa ASR server is LIVE

🌐 URLs:
   • Health check  : http://localhost:8000/
   • API docs (try it!) : http://localhost:8000/docs
   • Process audio : POST http://localhost:8000/process_audio
   • Process text  : POST http://localhost:8000/process_text


In [25]:
# ─── Test the server with a text request (no audio needed) ───────────────────
import requests

# Test 1: Health check
print("Test 1 — Health check:")
r = requests.get("http://localhost:8000/health")
print(f"  Status: {r.status_code}")
print(f"  Response: {r.json()}")
print()

# Test 2: Process text (the report's example phrase!)
print("Test 2 — Process the report's example phrase:")
r = requests.post(
    "http://localhost:8000/process_text",
    params={"text": "Khassna dwa dyal skhana f douar Tizgui"}
)
print(f"  Status: {r.status_code}")
import json
print(f"  Response:")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))
print()

# Test 3: A critical case
print("Test 3 — Critical case (snakebite):")
r = requests.post(
    "http://localhost:8000/process_text",
    params={"text": "lda3a dyal hanach f douar Ouirgane"}
)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

Test 1 — Health check:
  Status: 200
  Response: {'status': 'healthy', 'model_loaded': True}

Test 2 — Process the report's example phrase:
  Status: 200
  Response:
{
  "transcription": "Khassna dwa dyal skhana f douar Tizgui",
  "medication": "paracetamol",
  "urgency": "routine",
  "location": "Tizgui",
  "location_type": "village",
  "status": "ready_to_dispatch",
  "missing": [],
  "metadata": {
    "medication_source": "symptom",
    "medication_matched": "skhana",
    "location_confidence": "high"
  },
  "order_id": "AIRDWA-TEST-212452"
}

Test 3 — Critical case (snakebite):
{
  "transcription": "lda3a dyal hanach f douar Ouirgane",
  "medication": "antivenom",
  "urgency": "critical",
  "location": "Ouirgane",
  "location_type": "village",
  "status": "ready_to_dispatch",
  "missing": [],
  "metadata": {
    "medication_source": "symptom",
    "medication_matched": "lda3a",
    "location_confidence": "high"
  },
  "order_id": "AIRDWA-TEST-212452"
}


In [26]:
# ─── AirDwa Gradio Demo ──────────────────────────────────────────────────────
# Same logic as the FastAPI version, but with a UI judges can interact with

import gradio as gr
import numpy as np
import librosa
from datetime import datetime
import json

def airdwa_pipeline(audio_input, text_input):
    """
    Two input modes:
      - audio_input: recording from mic or uploaded file
      - text_input: type text directly (for fast testing)
    """
    # Decide which input to use (audio takes priority if both provided)
    if audio_input is not None:
        sample_rate, audio_array = audio_input
        
        # Normalize to float32 mono 16kHz
        audio_array = audio_array.astype(np.float32)
        if audio_array.ndim > 1:
            audio_array = audio_array.mean(axis=1)
        if np.abs(audio_array).max() > 1.0:
            audio_array = audio_array / 32768.0
        if sample_rate != 16000:
            audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)
        
        # Transcribe with Darija ASR
        audio_dict = {"array": audio_array, "sampling_rate": 16000}
        try:
            result = asr_pipeline(audio_dict, chunk_length_s=30)
            transcription = result["text"].strip()
        except Exception as e:
            return f"❌ ASR error: {e}", "—", "—", "—", "—", {"error": str(e)}
    
    elif text_input and text_input.strip():
        transcription = text_input.strip()
    
    else:
        return "⚠️ Please record audio or enter text", "—", "—", "—", "—", {}
    
    # Run extraction
    order = extract_order_info(transcription)
    order_id = f"AIRDWA-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    
    # Urgency badge with emoji
    urgency_display = {
        "critical": "🚨 CRITICAL — Dispatch immediately",
        "high":     "⚠️  HIGH — Priority delivery",
        "routine":  "✅ Routine",
    }.get(order["urgency"], "❓ Unknown")
    
    # Status badge
    status_display = (
        "🚁 READY TO DISPATCH" if order["status"] == "ready_to_dispatch"
        else f"⚠️ INCOMPLETE (missing: {', '.join(order['missing'])})"
    )
    
    # Build the dispatcher payload (what would be sent to teammates)
    dispatch_payload = {
        "order_id": order_id,
        "medication": order["medication"],
        "urgency": order["urgency"],
        "destination": order["location"],
        "source_transcription": transcription,
        "timestamp": datetime.now().isoformat(),
    }
    
    return (
        transcription,
        f"💊 {order['medication']}" if order["medication"] else "❌ Not detected",
        f"📍 {order['location']}" if order["location"] else "❌ Not detected",
        urgency_display,
        status_display,
        dispatch_payload,
    )


# ─── Build the interface ──────────────────────────────────────────────────────
with gr.Blocks(title="AirDwa | أير-دوا", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🚁 AirDwa | أير-دوا
        ### Darija voice → drone medical delivery
        
        Speak in Darija to request medication for a rural douar.  
        The AI extracts the symptom, identifies the medication needed, locates the village, 
        and prepares a dispatch order for the autonomous drone fleet.
        """
    )
    
    with gr.Row():
        # ─── LEFT: inputs ────────────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### 🎙️ Patient request")
            
            audio_in = gr.Audio(
                sources=["microphone", "upload"],
                type="numpy",
                label="Record or upload audio (Darija)",
            )
            
            gr.Markdown("**OR type text directly** (for quick testing without recording)")
            text_in = gr.Textbox(
                label="Darija text",
                placeholder="e.g. Khassna dwa dyal skhana f douar Tizgui",
                lines=2,
            )
            
            submit_btn = gr.Button("🔍 Process request", variant="primary", size="lg")
            
            gr.Markdown(
                """
                ### 💡 Try these example phrases:
                - `Khassna dwa dyal skhana f douar Tizgui` *(fever in Tizgui)*
                - `lda3a dyal hanach f douar Ouirgane` *(snakebite — critical!)*
                - `عندنا واحد المريض كيعاني من ربو في دوار إمليل` *(asthma in Imlil)*
                - `khassna insuline f Asni` *(insulin scheduled drop)*
                """
            )
        
        # ─── RIGHT: outputs ──────────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### 📋 Extracted order")
            
            transcription_out = gr.Textbox(label="📝 Darija transcription", interactive=False)
            medication_out = gr.Textbox(label="Medication", interactive=False)
            location_out = gr.Textbox(label="Destination", interactive=False)
            urgency_out = gr.Textbox(label="Urgency", interactive=False)
            status_out = gr.Textbox(label="Status", interactive=False)
            
            gr.Markdown("### 🔧 Dispatch payload (sent to drone coordinator)")
            json_out = gr.JSON(label="JSON sent to create_new_order()")
    
    # Wire button to function
    submit_btn.click(
        fn=airdwa_pipeline,
        inputs=[audio_in, text_in],
        outputs=[
            transcription_out,
            medication_out,
            location_out,
            urgency_out,
            status_out,
            json_out,
        ],
    )
    
    gr.Markdown(
        """
        ---
        **AirDwa** · HackAI 2026 · Built on Whisper-Darija + custom symptom-to-medication mapping
        """
    )

# ─── Launch ───────────────────────────────────────────────────────────────────
# Stop the FastAPI server first if it was running
try:
    stop_server()
except NameError:
    pass

demo.launch(
    share=True,        # 🌍 public URL — great for showing judges
    inbrowser=True,    # auto-opens in browser
    show_error=True,
)

/var/folders/7v/zj8flkfj6hq6v2q1cpwsdgwr0000gn/T/ipykernel_11461/209401312.py:81: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="AirDwa | أير-دوا", theme=gr.themes.Soft()) as demo:


🛑 Server stopping...
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://c85e89b7d62d6902cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/user/miniconda3/envs/hackai/lib/python3.10/site-packages/uvicorn/protocols/http/h11_impl.py", line 415, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "/Users/user/miniconda3/envs/hackai/lib/python3.10/site-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
  File "/Users/user/miniconda3/envs/hackai/lib/python3.10/site-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/user/miniconda3/envs/hackai/lib/python3.10/site-packages/starlette/applications.py", line 90, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Users/user/miniconda3/envs/hackai/lib/python3.10/site-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/Users/user/miniconda3/envs/hackai/lib/python3.10/site-packag